# WISER UWB Pilot Analysis

**QC gates interpretation.** This notebook's first job is to answer *"at what spatial and temporal
scale is this WISER dataset trustworthy?"* — only then does it present behavior, social, refuge, and
weather analyses, each restating the relevant caveat.

Reusable logic lives in `src/wiser_analysis_utils.py` (built on `wiser_io`, `time_utils`, `metrics`,
`plotting`). This notebook is a thin orchestration client. *Note:* the repo convention prefers marimo
notebooks, but a Jupyter `.ipynb` was explicitly requested.

Key facts (documented in `implementation_plan/2026-06-29-wiser-pilot-analysis.md`):
- **Units = inches.** WISER timestamps are Unix-ms **UTC**; local time is **EDT (UTC-4)**.
- The WISER frame has an **offset origin** and is **not** verified against the physical paddock — all
  boundary/edge/ROI results are in the WISER frame and labeled accordingly.
- The free-moving DB may be **live**; we re-query the snapshot at runtime and never hard-code counts.
- Distances in metres are converted to inches (1 in = 2.54 cm).

## 0. Configuration
Edit only this cell to rerun on a different WISER folder.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
try:                                  # inline images in Jupyter; no-op headless
    from IPython.display import Image, display
except Exception:
    def display(*a, **k): pass
    def Image(*a, **k): return None

# ---- paths (edit here) ----
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC))
import wiser_io, time_utils, metrics, plotting
import wiser_analysis_utils as w

DATA_DIR     = Path(r"D:\Wiser\data")
FREE_DB      = DATA_DIR / "1stcohort_2026.sqlite"      # free-moving session (may be LIVE)
FIXED_DB     = DATA_DIR / "tag_reports.sqlite"          # stationary baseline ("no movement")
GT_CSV       = PROJECT_ROOT / "configs" / "fixed_position_ground_truth.csv"
ROI_JSON     = PROJECT_ROOT / "configs" / "wiser_rois.json"
WEATHER_CSV  = Path(r"D:\weather_data\AWN-F8B3B78DEAC9-20260628-20260629.csv")
OUTPUT_ROOT  = Path(r"D:\Wiser_plot")                  # off C:, off git

# ---- analysis parameters ----
ACTIVE_SPEED_INPS = w.DEFAULT_ACTIVE_SPEED_INPS   # fallback; overwritten by the data-driven stationary speed-noise floor in Section B
MAX_SPEED_INPS    = w.DEFAULT_MAX_SPEED_INPS
SOCIAL_BIN_S      = 1.0
TZ_OFFSET_HOURS   = w.LOCAL_TZ_OFFSET_HOURS

OUT = w.make_output_dir(OUTPUT_ROOT)
FIG = OUT / "figures"
log_lines = [f"WISER pilot run — output {OUT}"]
print("Output folder:", OUT)

## 0a. Runtime snapshot & trustworthy-scale verdict

We capture the live snapshot (row count + timestamp range **at analysis time**) and, after QC below,
state the spatial/temporal scale at which the data can be trusted.

In [ ]:
snap = w.session_snapshot(FREE_DB)
print(json.dumps(snap, indent=2, default=str))
log_lines.append("Free-session snapshot: " + json.dumps(snap, default=str))
assert snap.get("n_rows", 0) > 0, "no rows in free-moving session"


## A. Load, timestamps, speed, validity flags (QC)

Read the free-moving session **read-only** (preserving `anchors_used`, `calculation_error`,
`battery_voltage`). Add UTC datetime + elapsed, raw & smoothed speed, and validity flags. **Flags only
— no rows are deleted.** The jitter floor used to justify thresholds comes from the stationary
baseline (Section B), computed first below.

In [ ]:
# --- stationary jitter floor first (needed to justify thresholds) ---
fixed = w.load_wiser_session(FIXED_DB)
fixed = time_utils.convert_timestamps(fixed)
fixed = time_utils.trim_last_n_minutes(fixed, minutes=10)   # tag-removal tail
fixed = w.add_speed(fixed)                                  # speeds on motionless tags = pure noise
noise = w.speed_noise_floor(fixed)                          # speed-noise floor (in/s)
ACTIVE_SPEED_INPS = round(noise['p99'], 1)                  # data-driven 'active' threshold
print(f"Speed-noise floor (stationary smoothed speed): median {noise['median']:.2f}, "
      f"p95 {noise['p95']:.2f}, p99 {noise['p99']:.2f} in/s  ->  active threshold {ACTIVE_SPEED_INPS} in/s")
log_lines.append(f"Speed-noise floor (stationary): {noise}; active threshold = {ACTIVE_SPEED_INPS} in/s; "
                 f"smoothed-speed cap = {w.MAX_PLAUSIBLE_SPEED_INPS} in/s")
gt = metrics.load_ground_truth(GT_CSV)
fixed_summary = metrics.compute_summary(fixed, ground_truth=gt)
jitter_floor_in = float(np.nanmedian(fixed_summary["rms_jitter"]))
print(f"Stationary jitter floor (median RMS): {jitter_floor_in:.2f} in "
      f"({jitter_floor_in*w.IN_TO_CM:.1f} cm)")
fixed_summary.to_csv(OUT / "fixed_position_error_summary.csv")
log_lines.append(f"Jitter floor (median RMS jitter) = {jitter_floor_in:.2f} in")
fixed_summary[[c for c in ['n_frames','rms_jitter','jitter_p95','rmse','median_error'] if c in fixed_summary.columns]]

In [ ]:
# --- free-moving session: load + derive ---
df = w.load_wiser_session(FREE_DB)
df = time_utils.convert_timestamps(df)
df = w.add_speed(df)

roi_cfg = w.load_rois(ROI_JSON)
boundary = roi_cfg.get("boundary") if roi_cfg else None
df = w.add_validity_flags(df, boundary=boundary, jitter_floor_in=jitter_floor_in,
                          max_speed_inps=MAX_SPEED_INPS)
df = w.apply_tag_cutoffs(df)             # exclude post-mortem/removed-tag fixes (e.g. Sova 12409)
n_cut = int(df.get('after_tag_cutoff', pd.Series(dtype=bool)).sum())
if n_cut:
    print(f'Per-tag cutoffs: {n_cut} post-cutoff fixes flagged invalid (e.g. deceased Sova/12409).')
    log_lines.append(f'Per-tag cutoffs applied (valid_until in rat_identities.csv): {n_cut} fixes invalidated.')
flags = w.flag_summary(df)
print(json.dumps(flags, indent=2))
log_lines.append("Flag summary: " + json.dumps(flags))
log_lines.append(f"Jump threshold = {MAX_SPEED_INPS} in/s; gap factor = {w.DEFAULT_GAP_FACTOR}x "
                 f"median dt; min anchors = {w.DEFAULT_MIN_ANCHORS}; "
                 f"bounds confirmed = {bool(boundary and boundary.get('confirmed'))}")

In [ ]:
# --- per-tag sampling rate from timestamp gaps ---
rate = (df.dropna(subset=['dt_s']).groupby('shortid')['dt_s']
        .agg(median_dt_s='median', n='size'))
rate['approx_hz'] = 1.0 / rate['median_dt_s']
rate

**Cleaned table** — raw rows retained, flags added (nothing deleted). Saved to `wiser_cleaned_pilot.csv`.

In [ ]:
clean_cols = ['ts_raw','datetime','elapsed_s','shortid','x','y','z','calculation_error',
              'anchors_used','battery_voltage','dt_s','speed_inps_raw','speed_inps_smooth',
              'step_in_smooth','gap_flag','jump_flag','low_anchor_flag',
              'outside_provisional_bounds','valid']
clean = df[[c for c in clean_cols if c in df.columns]].copy()
clean.to_csv(OUT / "wiser_cleaned_pilot.csv", index=False)
assert len(clean) == len(df), "cleaned row count must equal raw row count"
print("cleaned rows:", len(clean), "== raw rows:", len(df))

### A. QC figures — gaps, raw vs cleaned trajectories, speed

In [ ]:
w.plot_gap_histogram(df, save_path=FIG/"A1_gap_histogram.png"); display(Image(str(FIG/"A1_gap_histogram.png")))
w.plot_trajectories(df, save_path=FIG/"A2_traj_raw.png", valid_only=False, title_suffix="(RAW, all fixes)")
display(Image(str(FIG/"A2_traj_raw.png")))
w.plot_trajectories(df, save_path=FIG/"A3_traj_clean.png", valid_only=True, title_suffix="(valid only)")
display(Image(str(FIG/"A3_traj_clean.png")))
w.plot_speed_timeseries(df, save_path=FIG/"A4_speed.png", active_speed_inps=ACTIVE_SPEED_INPS)
display(Image(str(FIG/"A4_speed.png")))

## B. Static / fixed-position error (the "no movement" baseline)

The stationary tags give the precision floor. Jitter clouds below show per-tag spread; the summary CSV
holds bias / RMSE / percentiles vs ground truth. We compare the free-moving smoothed-speed activity to
this floor when interpreting movement.

In [ ]:
samp = fixed.iloc[::20].copy()  # stride-subsample for plotting (cloud shape unaffected)
w.plot_jitter_clouds_inches(samp, ground_truth=gt, save_path=FIG/"B1_jitter_clouds.png")
display(Image(str(FIG/"B1_jitter_clouds.png")))
print("Free-moving smoothed-speed median (in/s) vs jitter floor implied step:")
print(df.groupby('shortid')['speed_inps_smooth'].median())

## C. Paddock geometry (WISER frame — UNVERIFIED vs physical paddock)

Occupancy heatmaps and distance-to-edge use the WISER-frame boundary from `wiser_rois.json` (or the
provisional observed box). Out-of-bounds is **informational** unless the boundary is confirmed.

In [ ]:
extent = boundary['rect'] if boundary and 'rect' in boundary else w.observed_extent(df)
valid_df = df[df['valid']]
plotting.plot_occupancy_grid(valid_df, hour_label="full session (valid fixes)",
                             extent=tuple(extent), bin_inches=4.0,
                             save_path=FIG/"C1_occupancy.png")
display(Image(str(FIG/"C1_occupancy.png")))

d2e = w.distance_to_edge(valid_df, tuple(extent))
edge_margin = 24.0  # inches
near_edge_frac = float((d2e <= edge_margin).mean())
print(f"Fraction of valid fixes within {edge_margin:.0f} in of the (provisional) boundary: {near_edge_frac:.2f}")
log_lines.append(f"near-edge fraction (<= {edge_margin} in) = {near_edge_frac:.3f} "
                 f"[boundary confirmed={bool(boundary and boundary.get('confirmed'))}]")

## D. Behavior (movement & hourly activity)
Smoothed speed only (raw speed is jitter-inflated). Activity is **hourly**, no day/night split.

In [ ]:
mv = w.movement_summary(df, active_speed_inps=ACTIVE_SPEED_INPS)
mv.to_csv(OUT / "movement_summary.csv", index=False)

# occupancy summary per tag (valid fixes inside extent)
occ_rows = []
for tag, g in valid_df.groupby('shortid'):
    H, xe, ye = w.occupancy_hist(g, tuple(extent), bin_in=6.0)
    occ_rows.append({'shortid': tag, 'n_valid': int(len(g)),
                     'occupied_bins': int((H>0).sum()),
                     'top_bin_dwell_frac': float(H.max()/H.sum()) if H.sum() else np.nan})
occ = pd.DataFrame(occ_rows); occ.to_csv(OUT / "occupancy_summary.csv", index=False)

act = w.hourly_activity(df, active_speed_inps=ACTIVE_SPEED_INPS, tz_offset_hours=TZ_OFFSET_HOURS)
mv

## E. Social / spatial proximity
**Do not interpret as social preference until tracking quality and shelter locations are confirmed.** Proximity thresholds below the jitter floor are flagged unreliable.

In [ ]:
grid = w.resample_common_grid(df, bin_s=SOCIAL_BIN_S, valid_only=True)
dist_long = w.pairwise_distances(grid)
prox = w.proximity_summary(dist_long, jitter_floor_in=jitter_floor_in)
ci = w.clustering_index(dist_long)
print("Proximity reliability at each threshold:",
      prox.groupby('threshold_in')['reliable'].first().to_dict())
w.plot_pairwise_distance_heatmap(dist_long, save_path=FIG/"E1_pairwise_heatmap.png"); display(Image(str(FIG/"E1_pairwise_heatmap.png")))
w.plot_clustering_index(ci, save_path=FIG/"E2_clustering_index.png"); display(Image(str(FIG/"E2_clustering_index.png")))
prox

## F. Refuge / ROI analysis

ROIs come from `wiser_rois.json` if **confirmed** (placed via `scripts/place_wiser_rois.py`);
otherwise we show **candidate high-occupancy clusters** — explicitly *not* confirmed refuge/food/water.

In [ ]:
roi_confirmed = bool(roi_cfg and any(r.get('confirmed') for r in roi_cfg.get('rois', [])))
candidates = w.infer_candidate_zones(valid_df, extent=tuple(extent), k=4)
print("Candidate high-occupancy clusters (NOT confirmed refuge/food/water):")
display(candidates)

if roi_cfg:
    time_df, trans = w.roi_time_and_transitions(df, roi_cfg)
    label = "CONFIRMED ROIs" if roi_confirmed else "PROVISIONAL ROIs (placeholder — run the GUI)"
    print(f"ROI time / transitions using {label}:")
    display(time_df.head(20))
    w.plot_roi_transition_graph(roi_cfg, trans, time_df, save_path=FIG/"F1_roi_transitions.png")
    display(Image(str(FIG/"F1_roi_transitions.png")))
else:
    print("No wiser_rois.json — only candidate clusters shown.")
log_lines.append(f"ROIs confirmed = {roi_confirmed}")

## G. First-day acclimation
First 1 h / 3 h / rest. Descriptive only — ~12 h (one night) is **not** evidence of stable territory.

In [ ]:
accl = w.acclimation_windows(df)
display(accl.sort_values(['shortid','window']))
log_lines.append("Acclimation windows computed (first_1h/first_3h/rest_after_3h).")

## I. Hourly activity + weather (diel / light-related, EXPLORATORY — NOT circadian)

The session is < 24 h (one partial cycle), so this is **not** a circadian analysis. Temperature and
solar overlays and any activity-vs-temperature correlation are **exploratory**, aligned to weather by
wall-clock UTC (unverified to ~5 min).

In [ ]:
weather = w.load_weather(WEATHER_CSV)
merged = w.merge_activity_weather(act['group_hour'], weather)
merged.to_csv(OUT / "activity_weather_hourly.csv", index=False)
clock_summary = w.hourly_clock_summary(act['by_clock_per_tag'])   # between-rat mean +/- SD
clock_summary.to_csv(OUT / "hourly_activity_by_clock.csv", index=False)

w.plot_weather_timeseries(weather, save_path=FIG/"I1_weather.png"); display(Image(str(FIG/"I1_weather.png")))
w.plot_hourly_activity_by_clock(act['by_clock_per_tag'], weather=weather, save_path=FIG/"I2_hourly_by_clock.png")
display(Image(str(FIG/"I2_hourly_by_clock.png")))
corr = w.plot_activity_vs_temperature(merged, save_path=FIG/"I3_activity_vs_temp.png")
if Path(FIG/"I3_activity_vs_temp.png").exists(): display(Image(str(FIG/"I3_activity_vs_temp.png")))
print("Exploratory activity-vs-temperature:", corr)
log_lines.append(f"Activity-vs-temp (exploratory): {corr}")

## J. Leader-follower / route-following (candidate route use)

Following is defined as **time-lagged path reuse**, *not* proximity: for an ordered pair A→B, B follows A if B reaches A's earlier position after a positive lag while **both are moving** and their **headings align**. Scored on a 1 Hz grid of smoothed positions, validated against **circular-shift null controls**, and reported strictly as **candidate route-following** (not confirmed social following). The spatial threshold R is compared to the WISER stationary jitter floor before any close-following claim.

In [ ]:
# --- J. Leader-follower / route-following (candidate route use) ---
N_FOLLOW_SHUFFLES = 100                                  # circular-shift null controls per pair
MOVING_THR = w.grid_speed_noise_floor(fixed)             # conservative moving speed (grid p99 of stationary)
R_IN = w.follow_radius_in(jitter_floor_in)               # max(3 x jitter, 24 in)
t0 = df['datetime'].min()
print(f"Route-following: moving_thr={MOVING_THR:.2f} in/s, R={R_IN:.1f} in, "
      f"jitter_floor={jitter_floor_in:.1f} in, shuffles={N_FOLLOW_SHUFFLES}")

fgrid   = w.build_following_grid(df, moving_thr_inps=MOVING_THR)
fscores = w.follow_scores_all(fgrid, R=R_IN)
fpeaks  = w.following_peaks(fscores)
fasym   = w.following_asymmetry(fpeaks)
fnull   = w.following_null(fgrid, fpeaks, R=R_IN, n_shuffles=N_FOLLOW_SHUFFLES)

follow_tbl = (fpeaks.merge(fasym[['leader','follower','asymmetry']], on=['leader','follower'])
                    .merge(fnull, on=['leader','follower']))
follow_tbl['R_in'] = R_IN
follow_tbl['moving_thr_inps'] = MOVING_THR
follow_tbl['jitter_floor_in'] = jitter_floor_in
follow_tbl['reliable'] = bool(R_IN >= 3*jitter_floor_in)
follow_tbl['above_null'] = follow_tbl['peak_score'] > follow_tbl['shuffled_p95']
follow_tbl = follow_tbl.sort_values('peak_score', ascending=False)
follow_tbl.to_csv(OUT/'following_scores.csv', index=False)

# bouts for the strongest ordered pairs (raster + snippets)
top = fpeaks.dropna(subset=['peak_score']).sort_values('peak_score', ascending=False).head(8)
fevents = (pd.concat([w.following_events(fgrid, r.leader, r.follower, int(r.best_lag_s), R=R_IN)
                      for r in top.itertuples()], ignore_index=True) if len(top) else pd.DataFrame())
if not fevents.empty:
    fevents.to_csv(OUT/'following_events.csv', index=False)

w.plot_following_heatmap(fpeaks, save_path=FIG/'J1_follow_heatmap.png'); display(Image(str(FIG/'J1_follow_heatmap.png')))
w.plot_following_best_lag_heatmap(fpeaks, save_path=FIG/'J2_best_lag_heatmap.png'); display(Image(str(FIG/'J2_best_lag_heatmap.png')))
w.plot_following_asymmetry_heatmap(fasym, save_path=FIG/'J3_asymmetry_heatmap.png'); display(Image(str(FIG/'J3_asymmetry_heatmap.png')))
w.plot_following_lag_curves(fscores, fpeaks, null_df=fnull, save_path=FIG/'J4_lag_curves.png'); display(Image(str(FIG/'J4_lag_curves.png')))
w.plot_following_raster(fevents, save_path=FIG/'J5_raster.png')
if (FIG/'J5_raster.png').exists(): display(Image(str(FIG/'J5_raster.png')))
w.plot_following_snippets(fgrid, fevents, t0_datetime=t0, save_path=FIG/'J6_snippets.png')
if (FIG/'J6_snippets.png').exists(): display(Image(str(FIG/'J6_snippets.png')))

# candidate verdict (interpretation rules: real vs null, short-lag, asymmetry, R vs jitter floor)
n_above = int(follow_tbl['above_null'].sum()); n_pairs = len(follow_tbl)
strong = follow_tbl.head(5)
short_lag = float((strong['best_lag_s'] <= 7).mean())
asym_mag = float(strong['asymmetry'].abs().median())
if n_above >= n_pairs/2 and short_lag >= 0.6 and asym_mag >= 0.2:
    verdict = ('CANDIDATE leader-follower: short-lag follow scores exceed the circular-shift null '
               'and are directionally asymmetric.')
elif n_above >= n_pairs/2:
    verdict = ('Above-null path reuse but weak short-lag/asymmetry -> candidate SHARED ROUTE / corridor use.')
else:
    verdict = ('Real scores ~ shuffled controls -> shared corridor / shared route use, not leader-follower.')
follow_msg = (f'Route-following verdict (CANDIDATE, not confirmed social following): {verdict} '
              f'{n_above}/{n_pairs} ordered pairs above shuffled p95; strongest-pair median best-lag '
              f"{strong['best_lag_s'].median():.0f}s, |asymmetry| {asym_mag:.2f}. "
              f'R={R_IN:.0f} in vs jitter floor {jitter_floor_in:.1f} in (R>=3x floor: {bool(R_IN>=3*jitter_floor_in)}).')
print(follow_msg); log_lines.append(follow_msg)
follow_tbl.head(8)


## Final pilot conclusion + provenance
Auto-assembled from the QC numbers above; **review before citing.**

In [ ]:
def f2(x): return f"{x:.2f}"
valid_frac = flags['valid']['fraction']
jump_frac = flags['jump_flag']['fraction']
gap_frac = flags['gap_flag']['fraction']
reliab = prox.groupby('threshold_in')['reliable'].first().to_dict()

sections = {
 'data_quality': (f"Snapshot {snap.get('n_rows'):,} rows, {snap.get('duration_h')} h "
    f"({snap.get('min_utc')} -> {snap.get('max_utc')} UTC). Valid fraction {f2(valid_frac)}; "
    f"dropouts {f2(gap_frac)}, impossible jumps {f2(jump_frac)}. Stationary jitter floor "
    f"{jitter_floor_in:.1f} in ({jitter_floor_in*w.IN_TO_CM:.0f} cm)."),
 'tracking_reliability': "Per-tag rate ~%.2f Hz; see fixed_position_error_summary.csv for per-tag jitter/RMSE. "
    "All 6 tags present and sampled continuously." % (rate['approx_hz'].median()),
 'artifacts': (f"Impossible jumps {f2(jump_frac)} of fixes (raw speed > {MAX_SPEED_INPS:.0f} in/s); "
    f"dropouts {f2(gap_frac)}; low-anchor {f2(flags['low_anchor_flag']['fraction'])}. "
    "Raw frame-to-frame speed is jitter-inflated; activity uses smoothed speed."),
 'activity': ("Per-tag distance/active-fraction in movement_summary.csv; hourly pattern in "
    "activity_weather_hourly.csv. 'Active' = smoothed speed above the data-driven stationary "
    "speed-noise floor (%.1f in/s, the p99 of motionless-tag speed); below that, motion is "
    "indistinguishable from jitter. Smoothed speed is capped at %.0f in/s (~1.5 m/s) as artifact."
    % (ACTIVE_SPEED_INPS, w.MAX_PLAUSIBLE_SPEED_INPS)),
 'spatial_structure': (f"Occupancy concentrates in a few clusters (see candidate zones); "
    f"{f2(near_edge_frac)} of valid fixes near the provisional boundary. WISER frame is UNVERIFIED "
    "vs the physical paddock, so edge/wall claims are provisional."),
 'refuge_evidence': ("Candidate high-occupancy clusters identified but NOT confirmed as refuge/food/water; "
    "confirm by placing ROIs with scripts/place_wiser_rois.py. ROIs confirmed = %s." % roi_confirmed),
 'ready_for_formal': ("Tracking is usable at the scale of occupancy/among-tag comparison. Proximity "
    "reliability by threshold: %s. NOT yet ready for fine social-distance or absolute-speed claims "
    "until the jitter floor and shelter locations are confirmed." % reliab),
 'fix_before_next': ("1) Place real ROIs + boundary (GUI) to georeference the WISER frame. "
    "2) Record a fresh stationary test per session for the jitter floor. "
    "3) Capture >=24 h for any circadian claim. "
    "4) Verify WISER<->paddock and WISER<->weather clock alignment."),
}
report = w.build_pilot_report(sections)
print(report)
(OUT / "pilot_conclusion.txt").write_text(report, encoding="utf-8")

# QC summary CSV (per tag) and provenance
tag_qc = fixed_summary[[c for c in ['n_frames','rms_jitter','jitter_p95'] if c in fixed_summary.columns]].copy()
tag_qc.to_csv(OUT / "tag_qc_summary.csv")
w.write_filtering_log(OUT, log_lines)
w.write_run_manifest(OUT, {
    "free_session": snap,
    "fixed_baseline": str(FIXED_DB),
    "weather_file": str(WEATHER_CSV),
    "excluded": ["test20260622.csv (overlaps tag_reports.sqlite)"],
    "jitter_floor_in": jitter_floor_in,
    "speed_noise_floor_inps": noise,
    "thresholds": {"max_speed_inps": MAX_SPEED_INPS, "active_speed_inps": ACTIVE_SPEED_INPS,
                   "smoothed_speed_cap_inps": w.MAX_PLAUSIBLE_SPEED_INPS,
                   "speed_window_s": w.DEFAULT_SPEED_WINDOW_S,
                   "gap_factor": w.DEFAULT_GAP_FACTOR, "min_anchors": w.DEFAULT_MIN_ANCHORS,
                   "social_bin_s": SOCIAL_BIN_S},
    "route_following": {"R_in": R_IN, "moving_thr_inps": MOVING_THR,
                         "lags_s": [1, 30], "n_shuffles": N_FOLLOW_SHUFFLES,
                         "n_pairs_above_null": int(follow_tbl["above_null"].sum())},
    "coordinate_note": "WISER native inches, offset origin, UNVERIFIED vs physical paddock",
    "alignment_note": "WISER<->weather by wall-clock UTC, unverified (~5 min)",
})
print("\nAll outputs written to:", OUT)